# Intégration de logos pour la détection des bacs de tri

Le but est d'avoir trois logos (un pour chaque catégorie de déchets) 

## Import

In [2]:
import cv2
import matplotlib.pyplot as plt
import os
import numpy as np
import albumentations as A
import time
from tqdm import tqdm
import random
from pathlib import Path
import glob

## PATH

In [3]:
LOGO_DIR = Path("../data/logos")
BG_DIR = Path("../dataset/TACO-master/data/images")
OUTPUT_DIR = Path("../dataset/synthetic_logos_yolo")

# Création des dossiers de sortie pour YOLO
(OUTPUT_DIR / "train/images").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "train/labels").mkdir(parents=True, exist_ok=True)

# On crée la liste des fichiers d'arrière-plan (obligatoire pour le random.choice)
all_backgrounds = list(BG_DIR.glob("*.jpg"))

print(f"Nombre de fonds disponibles : {len(all_backgrounds)}")

Nombre de fonds disponibles : 1500


## Charger les logos de tri

In [4]:
# Dictionnaire pour tes 3 catégories
logos_pilotes = {
    "verre": cv2.imread(str(LOGO_DIR / "logo_verre.png"), cv2.IMREAD_UNCHANGED),
    "jaune": cv2.imread(str(LOGO_DIR / "logo_jaune.png"), cv2.IMREAD_UNCHANGED),
    "noir":  cv2.imread(str(LOGO_DIR / "logo_noir.png"), cv2.IMREAD_UNCHANGED)
}

Vérification, si canaux=4 alors on a bien un fond transparent

In [5]:
for nom, img in logos_pilotes.items():
    if img is None:
        print(f"⚠️ Erreur : Le logo '{nom}' n'a pas été trouvé. Vérifie le chemin !")
    else:
        print(f"✅ Logo '{nom}' chargé : {img.shape[1]}x{img.shape[0]} pixels (Canaux: {img.shape[2]})")

✅ Logo 'verre' chargé : 2816x1536 pixels (Canaux: 4)
✅ Logo 'jaune' chargé : 225x225 pixels (Canaux: 4)
✅ Logo 'noir' chargé : 177x148 pixels (Canaux: 4)


## Modification sur les logos

In [6]:
def apply_scl_transform(logo, target_width):
    # On redimensionne le logo pour qu'il fasse, par exemple, 
    # entre 10% et 30% de la largeur du fond (target_width)
    scale = random.uniform(0.1, 0.3)
    new_w = int(target_width * scale)
    
    h, w = logo.shape[:2]
    new_h = int(h * (new_w / w))
    
    logo = cv2.resize(logo, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
    # On garde le reste des transformations (rotation, couleur)
    rows, cols = logo.shape[:2]
    M_rot = cv2.getRotationMatrix2D((cols/2, rows/2), random.randint(-15, 15), 1)
    logo = cv2.warpAffine(logo, M_rot, (cols, rows), borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0,0))
    
    r = random.uniform(0.7, 1.3)
    logo[:, :, :3] = np.clip(logo[:, :, :3].astype(np.float32) * r, 0, 255).astype(np.uint8)
    
    return logo

## Transformation SCL (Synthetic Context Logo)

In [7]:
def create_synthetic_image(background_path, logo_img):
    bg = cv2.imread(str(background_path))
    if bg is None: return None, None
    bg_h, bg_w = bg.shape[:2]

    # Appliquer les transformations SCL au logo (Géométrie + Couleur), bg_w est utilisé pour redimensionner le logo à une taille réaliste par rapport au fond
    logo = apply_scl_transform(logo_img, bg_w) 
    l_h, l_w = logo.shape[:2]

    # Sécurité : si après transformation le logo est toujours trop grand (rare)
    if l_w >= bg_w or l_h >= bg_h:
        return None, None

    x = random.randint(0, bg_w - l_w)
    y = random.randint(0, bg_h - l_h)

    # 3. Fusionner le logo avec le fond en utilisant le canal Alpha
    alpha_mask = logo[:, :, 3] / 255.0
    for c in range(3):
        bg[y:y+l_h, x:x+l_w, c] = (1.0 - alpha_mask) * bg[y:y+l_h, x:x+l_w, c] + alpha_mask * logo[:, :, c]

    # 4. Coordonnées YOLO
    x_center = (x + l_w / 2) / bg_w
    y_center = (y + l_h / 2) / bg_h
    w_norm = l_w / bg_w
    h_norm = l_h / bg_h
    
    return bg, [x_center, y_center, w_norm, h_norm]

## Génération des images

In [8]:
# Paramètres de ton projet
N_IMAGES_PER_CLASS = 5 #100 pour un dataset plus grand voir 500 pour un dataset très grand
# Associe chaque nom à un index (0, 1, 2) pour YOLO
class_map = {"verre": 0, "jaune": 1, "noir": 2}

for label_name, logo_orig in logos_pilotes.items():
    print(f"Génération de la classe : {label_name}...")
    class_id = class_map[label_name]
    
    for i in range(N_IMAGES_PER_CLASS):
        # 1. Choisir un fond TACO au hasard 
        bg_path = random.choice(all_backgrounds)
        img_synth, bbox = create_synthetic_image(bg_path, logo_orig)
        
        if img_synth is None: continue
        
        # Nom de fichier unique
        file_name = f"{label_name}_synth_{i}"
        
        # 2. Sauvegarder l'image
        cv2.imwrite(str(OUTPUT_DIR / "train/images" / f"{file_name}.jpg"), img_synth)
        
        # 3. Sauvegarder le label YOLO (.txt)
        # Format : <class_id> <x_center> <y_center> <width> <height>
        with open(OUTPUT_DIR / "train/labels" / f"{file_name}.txt", "w") as f:
            f.write(f"{class_id} {bbox[0]} {bbox[1]} {bbox[2]} {bbox[3]}")

print("✅ Dataset synthétique généré avec succès !")

Génération de la classe : verre...
Génération de la classe : jaune...
Génération de la classe : noir...
✅ Dataset synthétique généré avec succès !


## Création du .yaml

In [9]:
import yaml

base_path = '../dataset/synthetic_logos_yolo'

data_yaml = {
        'path': base_path,
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'nc': len(class_map),
        'names': class_map,
    }

yaml_file_path = f'{base_path}/data_logos.yaml'
with open(yaml_file_path, 'w') as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=False)

print(f"Fichier YAML créé ici : {yaml_file_path}")

Fichier YAML créé ici : ../dataset/synthetic_logos_yolo/data_logos.yaml
